# 下準備

In [ ]:
# ライブラリのインポート
import os
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models, utils, optimizers, losses, metrics
from matplotlib import pyplot as plt
import numpy as np
from tqdm import tqdm
import cv2
import csv
from google.colab.patches import cv2_imshow
from IPython.display import Image
from google.colab import drive
drive.mount('/content/drive')

# 画像の読み込み関数
def read_and_preprocess(image_path):
    read = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(read, channels = 3)
    image = tf.image.resize(image, [224,224])
    image = tf.cast(image, tf.float32) / 255.0
    return image

# 画像とラベルを返す
def process(path, label):
    image = read_and_preprocess(path)
    label = tf.one_hot(label, depth=7)
    return image, label

# データセットを作成する
def generate_image_dataset(image_paths, labels, is_training=True):
    tmp = list(zip(image_paths, labels))
    np.random.shuffle(tmp)
    image_paths, labels = zip(*tmp)
    image_paths, labels = list(image_paths), list(labels)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    dataset = dataset.map(process, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        dataset = dataset.shuffle(1000)
    dataset = dataset.batch(32)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# 画像のパスとラベルのペアを作る
def get_image_paths_and_labels(base_dir):
    image_paths = []
    labels = []
    for class_name in sorted(os.listdir(base_dir)):
        class_path = os.path.join(base_dir, class_name)
        if os.path.isdir(class_path):
            for fname in os.listdir(class_path):
                image_paths.append(os.path.join(class_path, fname))
                labels.append(label2number[class_name])
    return image_paths, labels

model_path = "drive/MyDrive/CNN_pic/model/"
csv_path = "drive/MyDrive/CNN_pic/csv/"
# モデルを保存するフォルダの作成
os.makedirs(model_path, exist_ok=True)
# csvを保存するフォルダの作成
os.makedirs(csv_path, exist_ok=True)

# データセットを作成するときに、フォルダ名をIDに変換するための辞書
label2number = {"angry":0, "disgust":1, "fear":2, "happy":3, "sad":4, "surprise":5, "neutral":6}
# 予測結果を数字から文字に変換するための辞書
number2label = {0:"angry", 1:"disgust", 2:"fear", 3:"happy", 4:"sad", 5:"surprise", 6:"neutral"}

データセット  
https://www.kaggle.com/datasets/msambare/fer2013

In [ ]:
# kaggleからデータセットを読み込み
if not os.path.exists("/kaggle/input/fer2013"):
    print("データのダウンロードします")
    path = kagglehub.dataset_download("msambare/fer2013")
    print(path)
    print("ダウンロードが完了しました")
else:
    path = "/kaggle/input/fer2013"

print("データセットを作成します")
# 画像のパスとラベルをリストに追加
train_images, train_labels = get_image_paths_and_labels(path+"/train")
test_images, test_labels = get_image_paths_and_labels(path+"/test")

# データセットを作成
train_dataset = generate_image_dataset(train_images, train_labels, is_training=True)
test_dataset = generate_image_dataset(test_images, test_labels, is_training=False)
print("作成完了しました")

In [ ]:
# データを確認
image, label = next(iter(train_dataset))

print("trainデータの総数 :", len(train_images))
print("testデータの総数 :", len(train_images))
print("画像の大きさ :", image[0].shape)
print("ラベル(one-hotベクトル)の形 :", np.array(label)[0])

In [ ]:
# 画像を確認してみよう
for i in image[0]:
    for j in i:
        print(np.array(j), end=", ")
    print()

In [ ]:
# ランダムに画像を表示
fig = plt.figure()
for i in range(5):
    idx = np.random.randint(0, len(train_images)-1, 1)
    image = read_and_preprocess(train_images[idx[0]])
    ax = fig.add_subplot(1, 5, i+1)
    ax.imshow(image)
    ax.axis("off")
plt.show()

# AlexNet

In [ ]:
def AlexNet_build():
    input = layers.Input(shape=(224, 224, 3))

    # Block 1
    # 畳み込み
    x = layers.ZeroPadding2D(padding=2)(input)
    x =
    # 活性化関数
    x =
    # 最大値プーリング
    x =

    # Block 2
    # 畳み込み
    x = layers.ZeroPadding2D(padding=2)(x)
    x =
    # 活性化関数
    x =
    # 最大値プーリング
    x =

    # Block 3
    # 畳み込み
    x = layers.ZeroPadding2D(padding=1)(x)
    x =
    # 畳み込み
    x = layers.ZeroPadding2D(padding=1)(x)
    x =
    # 畳み込み
    x = layers.ZeroPadding2D(padding=1)(x)
    x =
    # 最大値プーリング
    x =

    # FC層
    # 1次元化
    x =
    # 全結合層
    x =
    # 活性化関数
    x =
    # ドロップアウト
    x =
    # 全結合層
    x =
    # 活性化関数
    x =
    # ドロップアウト
    x =
    # 全結合層
    output =

    return models.Model(input, output)

alexnet = AlexNet_build()
alexnet.summary()

In [ ]:
epochs = 10
alexnet.compile(loss = losses.CategoricalCrossentropy(), # lossの指定
              optimizer = optimizers.Adam(learning_rate = 0.0001), # optimizerの指定
              metrics = [metrics.CategoricalAccuracy()])
alexnet_history = alexnet.fit(train_dataset, validation_data=test_dataset, epochs=epochs)
alexnet.save(f"{model_path}/alexnet.keras")
with open(f'{csv_path}alexnet.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(alexnet_history.history.values())

# VGG11

In [ ]:
def VGG11_build():
    input = layers.Input(shape=(224, 224, 3))

    # Block 1
    # 畳み込み
    x =
    # 活性化関数
    x =
    # maxプーリング
    x =

    # Block 2
    # 畳み込み
    x =
    # 活性化関数
    x =
    # maxプーリング
    x =

    # Block 3
    # 畳み込み
    x =
    # 活性化関数
    x =
    # 畳み込み
    x =
    # 活性化関数
    x =
    # maxプーリング
    x =

    # Block 4
    # 畳み込み
    x =
    # 活性化関数
    x =
    # 畳み込み
    x =
    # 活性化関数
    x =
    # maxプーリング
    x =

    # Block 5
    # 畳み込み
    x =
    # 活性化関数
    x =
    # 畳み込み
    x =
    # 活性化関数
    x =
    # maxプーリング
    x =

    # FC層
    # 1次元化
    x =
    # 全結合層
    x =
    # 活性化関数
    x =
    # 全結合層
    x =
    # 活性化関数
    x =
    # 出力層
    output =

    return models.Model(input, output)

# モデルの作成と要約
VGG11 = VGG11_build()
VGG11.summary()

In [ ]:
epochs = 10
VGG11.compile(loss = losses.CategoricalCrossentropy(), # lossの指定
              optimizer = optimizers.Adam(learning_rate = 0.0001), # optimizerの指定
              metrics = [metrics.CategoricalAccuracy()])
VGG11_history = VGG11.fit(train_dataset, validation_data=test_dataset, epochs=epochs)
VGG11.save(f"{model_path}/VGG11.keras")
with open(f'{csv_path}vgg11.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(VGG11_history.history.values())

# ResNet-18

In [ ]:
# 残差ブロック（BasicBlock）
def basic_block(x, filters, stride=1):
    shortcut = x

    # 畳み込み1
    x = layers.Conv2D(filters, kernel_size=3, strides=stride, padding='same', use_bias=False)(x)
    # バッチノーマライゼーション
    x =
    # 活性化関数
    x =

    # 畳み込み2
    x = layers.Conv2D(filters, kernel_size=3, strides=1, padding='same', use_bias=False)(x)
    # バッチノーマライゼーション
    x =

    # shortcut（必要なら1x1で形状調整）
    if shortcut.shape[-1] != filters or shortcut.shape[1] != x.shape[1] or shortcut.shape[2] != x.shape[2]:
        shortcut = layers.Conv2D(filters, kernel_size=1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # 残差加算と活性化
    x = layers.ReLU()(x + shortcut)
    return x

# ブロックをまとめる関数（2回 basic_block を繰り返す）
def block_layer(x, filters, blocks, stride):
    # 最初のブロックはストライド付き（サイズ変更）
    x = basic_block(x, filters, stride=stride)
    for _ in range(1, blocks):
        x = basic_block(x, filters, stride=1)
    return x

# ResNet-18 本体
def ResNet18_build():
    inputs = layers.Input(shape=(224, 224, 3))

    # 初期層
    x = layers.Conv2D(64, kernel_size=7, strides=2, padding='same', use_bias=False)(inputs)
    # バッチノーマライゼーション
    x =
    # 活性化関数
    x =
    # maxプーリング
    x = layers.MaxPool2D(pool_size=3, strides=2, padding='same')(x)

    # 各ブロック層（ResNet18構成）
    x = block_layer(x, filters=??, blocks=2, stride=1) # conv2_x
    x = block_layer(x, filters=??, blocks=2, stride=2) # conv3_x
    x = block_layer(x, filters=??, blocks=2, stride=2) # conv4_x
    x = block_layer(x, filters=??, blocks=2, stride=2) # conv5_x

    # 分類層
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(7, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs)

# モデルの構築と表示
ResNet18 = ResNet18_build()
ResNet18.summary()

In [ ]:
epochs = 10
ResNet18.compile(loss = losses.CategoricalCrossentropy(), # lossの指定
              optimizer = optimizers.Adam(learning_rate = 0.0001), # optimizerの指定
              metrics = [metrics.CategoricalAccuracy()])
ResNet18_history = ResNet18.fit(train_dataset, validation_data=test_dataset, epochs=epochs)
ResNet18.save(f"{model_path}/Resnet18.keras")
with open(f'{csv_path}resnet18.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerows(ResNet18_history.history.values())

# グラフで確認

In [ ]:
with open(csv_path+"alexnet.csv", "r") as f:
    alexnet_csv = [list(map(float, i.split(","))) for i in f.read().split()]

# alexnet
fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("accuracy vs epochs")
ax.plot(range(10), alexnet_csv[0], label="accuracy")
ax.plot(range(10), alexnet_csv[2], label="val accuracy")
ax.set_xlabel("epochs")
ax.set_ylabel("accuracy")
ax.legend()
plt.show()

fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("loss vs epochs")
ax.plot(range(10), alexnet_csv[1], label="loss")
ax.plot(range(10), alexnet_csv[3], label="val loss")
ax.set_xlabel("epochs")
ax.set_ylabel("loss")
ax.legend()
plt.show()

In [ ]:
with open(csv_path+"vgg11.csv", "r") as f:
    vgg11_csv = [list(map(float, i.split(","))) for i in f.read().split()]

# vgg11
fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("accuracy vs epochs")
ax.plot(range(10), vgg11_csv[0], label="accuracy")
ax.plot(range(10), vgg11_csv[2], label="val accuracy")
ax.set_xlabel("epochs")
ax.set_ylabel("accuracy")
ax.legend()
plt.show()

fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("loss vs epochs")
ax.plot(range(10), vgg11_csv[1], label="loss")
ax.plot(range(10), vgg11_csv[3], label="val loss")
ax.set_xlabel("epochs")
ax.set_ylabel("loss")
ax.legend()
plt.show()

In [ ]:
with open(csv_path+"resnet18.csv", "r") as f:
    resnet18_csv = [list(map(float, i.split(","))) for i in f.read().split()]

# ResNet18
fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("accuracy vs epochs")
ax.plot(range(10), resnet18_csv[0], label="accuracy")
ax.plot(range(10), resnet18_csv[2], label="val accuracy")
ax.set_xlabel("epochs")
ax.set_ylabel("accuracy")
ax.legend()
plt.show()

fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("loss vs epochs")
ax.plot(range(10), resnet18_csv[1], label="loss")
ax.plot(range(10), resnet18_csv[3], label="val loss")
ax.set_xlabel("epochs")
ax.set_ylabel("loss")
ax.legend()
plt.show()

In [ ]:
with open(csv_path+"alexnet.csv", "r") as f:
    alexnet_csv = [list(map(float, i.split(","))) for i in f.read().split()]
with open(csv_path+"vgg11.csv", "r") as f:
    vgg11_csv = [list(map(float, i.split(","))) for i in f.read().split()]
with open(csv_path+"resnet18.csv", "r") as f:
    resnet18_csv = [list(map(float, i.split(","))) for i in f.read().split()]

# すべてを重ねて確認
# accuracy
fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("accuracy vs epochs")
try:ax.plot(range(10), alexnet_csv[0], label="alexnet")
except:pass
try:ax.plot(range(10), vgg11_csv[0], label="VGG11")
except:pass
try:ax.plot(range(10), resnet18_csv[0], label="ResNet")
except:pass
ax.set_xlabel("epochs")
ax.set_ylabel("accuracy")
ax.legend()
plt.show()

# loss
fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("loss vs epochs")
try:ax.plot(range(10), alexnet_csv[1], label="alexnet")
except:pass
try:ax.plot(range(10), vgg11_csv[1], label="VGG11")
except:pass
try:ax.plot(range(10), resnet18_csv[1], label="ResNet")
except:pass
ax.set_xlabel("epochs")
ax.set_ylabel("loss")
ax.legend()
plt.show()

# val accuracy
fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("val accuracy vs epochs")
try:ax.plot(range(10), alexnet_csv[2], label="alexnet")
except:pass
try:ax.plot(range(10), vgg11_csv[2], label="VGG11")
except:pass
try:ax.plot(range(10), resnet18_csv[2], label="ResNet")
except:pass
ax.set_xlabel("epochs")
ax.set_ylabel("val accuracy")
ax.legend()
plt.show()

# val loss
fig = plt.figure()
ax = fig.add_subplot()
ax.set_title("val loss vs epochs")
try:ax.plot(range(10), alexnet_csv[3], label="alexnet")
except:pass
try:ax.plot(range(10), vgg11_csv[3], label="VGG11")
except:pass
try:ax.plot(range(10), resnet18_csv[3], label="ResNet")
except:pass
ax.set_xlabel("epochs")
ax.set_ylabel("val loss")
ax.legend()
plt.show()